In [4]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S4_THRESHOLD = 0.00
MIN_EVENT_POINTS = 5  # ignore tiny spikes
MAX_GAP = pd.Timedelta("2 min")

outdir = Path("/Users/dal674840/Downloads/20240629/scintillation_event_june_concat")
outdir.mkdir(exist_ok=True)
df = pd.read_parquet(
    "/Users/dal674840/Downloads/20240629/scintpi3_20240629_day_19.2235E_34.4244S_lvl3.pq"
)
df = df.copy()
df["datetime"] = pd.to_datetime(df["datetime"])


df = df[df["elev"] >= 20]

for (cons, svid), sat in df.groupby(["cons", "svid"]):
    sat = sat.sort_values("datetime").reset_index(drop=True)

    if "sigma_phi_1" in sat.columns:
        sat["sigma_phi_1_plot"] = sat["sigma_phi_1"]
        if "sigma_phi_quality_flag_1" in sat.columns:
            sat.loc[sat["sigma_phi_quality_flag_1"] != 0, "sigma_phi_1_plot"] = np.nan
        sat["sigma_phi_1_plot"] = sat["sigma_phi_1_plot"].interpolate(
            method="linear", limit_direction="both"
        )

    if "sigma_phi_2" in sat.columns:
        sat["sigma_phi_2_plot"] = sat["sigma_phi_2"]
        if "sigma_phi_quality_flag_2" in sat.columns:
            sat.loc[sat["sigma_phi_quality_flag_2"] != 0, "sigma_phi_1_plot"] = np.nan
        sat["sigma_phi_2_plot"] = sat["sigma_phi_2_plot"].interpolate(
            method="linear", limit_direction="both"
        )

    mask = (sat["s4_1"] >= S4_THRESHOLD) | (sat["s4_2"] >= S4_THRESHOLD)

    idx = np.where(mask)[0]

    if len(idx) == 0:
        continue

    groups = [[idx[0]]]

    for i in idx[1:]:
        dt = sat.loc[i, "datetime"] - sat.loc[groups[-1][-1], "datetime"]

        if dt <= MAX_GAP:
            groups[-1].append(i)
        else:
            groups.append([i])

    event_number = 1

    for g in groups:
        if len(g) < MIN_EVENT_POINTS:
            continue

        event = sat.iloc[g].copy()

        start = event["datetime"].iloc[0]
        end = event["datetime"].iloc[-1]

        fig, ax = plt.subplots(
            6, 1, figsize=(10, 12), sharex=True, constrained_layout=True
        )

        ax[0].plot(event["datetime"], event["tec_cph12"], "k", lw=1.5)
        ax[0].set_ylabel("TEC (TECU)")
        ax[0].legend(["TEC"])

        ax[1].plot(event["datetime"], event["snr1"], color="red", lw=1, label="SNR1")

        ax[1].plot(event["datetime"], event["snr2"], color="blue", lw=1, label="SNR2")

        ax[1].set_ylabel("C/N0")
        ax[1].legend()

        ax[2].plot(event["datetime"], event["s4_1"], color="red", lw=1.8, label="S4 L1")

        ax[2].plot(
            event["datetime"], event["s4_2"], "--", color="blue", lw=1.8, label="S4 L2"
        )

        ax[2].set_ylim(0, 1)
        ax[2].set_ylabel("S4")
        ax[2].legend()

        if "sigma_phi_1_plot" in event.columns:
            ax[3].plot(
                event["datetime"],
                event["sigma_phi_1_plot"],
                color="red",
                lw=1.5,
                label="Sigma-$\\phi$ L1",
            )

        if "sigma_phi_2_plot" in event.columns:
            ax[3].plot(
                event["datetime"],
                event["sigma_phi_2_plot"],
                color="blue",
                lw=1.5,
                label="Sigma-$\\phi$ L2",
            )

        ax[3].set_ylabel(r"$\sigma_\phi$ (rad)")
        ax[3].set_ylim(0, 1.0)
        ax[3].legend()

        ax[4].plot(event["datetime"], event["elev"], "k", lw=1.5)

        ax[4].set_ylabel("Elevation (°)")
        ax[4].legend(["Elevation"])

        ax[5].plot(event["datetime"], event["azim"], "k", lw=1.5)

        ax[5].set_ylabel("Azimuth (°)")
        ax[5].set_xlabel("Time (UTC)")
        ax[5].legend(["Azimuth"])

        locator = mdates.AutoDateLocator()
        formatter = mdates.DateFormatter("%H:%M:%S")

        for a in ax:
            a.grid(alpha=0.3)
            a.xaxis.set_major_locator(locator)
            a.xaxis.set_major_formatter(formatter)

        plt.suptitle(
            f"Scintillation event on {start:%Y-%m-%d %H:%M:%S} "
            f"for satellite {cons}{int(svid)}"
        )

        plt.savefig(
            outdir / f"scintillation_event_{cons}{int(svid)}_{start:%Y%m%d_%H%M%S}.png",
            dpi=300,
            bbox_inches="tight",
        )

        plt.close()

        event_number += 1

print("Done.")

Done.


In [ ]:
import pandas as pd

df = pd.read_parquet(
    "/home/dal674840/scratch/bin_file/concat_lvl3/tx01_DOY194_20-24UTC_lvl3.parquet"
)
print(df.columns)